In [0]:
from pyspark.sql import functions as F

# Base volume path
base_path = "/Volumes/workspace/default/indian_ecommerce_sales_analytics/raw"

# Read Customers
customers_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/customers.csv")
)

# Read Products
products_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/products.csv")
)

# Read Sales
sales_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/sales.csv")
)

In [0]:
customers_bronze.printSchema()
products_bronze.printSchema()
sales_bronze.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Age_Group: string (nullable = true)
 |-- Date_of_Birth: date (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Pincode: integer (nullable = true)
 |-- Registration_Date: date (nullable = true)
 |-- Customer_Tier: string (nullable = true)
 |-- Total_Orders: integer (nullable = true)
 |-- Total_Spent: double (nullable = true)

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Original_Price: double (nullable = true)
 |-- Discount_Percent: integer (nullable = true)
 |-- Discount_Amount: double (nullable = true)
 |-- Selling_Price: double (nullable = true)
 |-- Stock_Quantity: integ

Creating delta Tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import functions as F

# Customers Bronze Metadata
customers_bronze = (
    customers_bronze
    .withColumn("_source_file", F.lit("customers.csv"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Products Bronze Metadata
products_bronze = (
    products_bronze
    .withColumn("_source_file", F.lit("products.csv"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Sales Bronze Metadata
sales_bronze = (
    sales_bronze
    .withColumn("_source_file", F.lit("sales.csv"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)
customers_bronze.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Age_Group: string (nullable = true)
 |-- Date_of_Birth: date (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Pincode: integer (nullable = true)
 |-- Registration_Date: date (nullable = true)
 |-- Customer_Tier: string (nullable = true)
 |-- Total_Orders: integer (nullable = true)
 |-- Total_Spent: double (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingestion_timestamp: timestamp (nullable = false)



In [0]:
(
    customers_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.indian_ecommerce_sales_analytics.bronze_customers"
    )
)
(
    products_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.indian_ecommerce_sales_analytics.bronze_products"
    )
)
(
    sales_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.indian_ecommerce_sales_analytics.bronze_sales"
    )
)

In [0]:
%sql
SHOW TABLES IN workspace.indian_ecommerce_sales_analytics;

database,tableName,isTemporary
indian_ecommerce_sales_analytics,bronze_customers,false
indian_ecommerce_sales_analytics,bronze_products,false
indian_ecommerce_sales_analytics,bronze_sales,false


In [0]:
%sql
SELECT
    'bronze_customers' AS table_name,
    COUNT(*) AS row_count
FROM workspace.indian_ecommerce_sales_analytics.bronze_customers

UNION ALL

SELECT
    'bronze_products',
    COUNT(*)
FROM workspace.indian_ecommerce_sales_analytics.bronze_products

UNION ALL

SELECT
    'bronze_sales',
    COUNT(*)
FROM workspace.indian_ecommerce_sales_analytics.bronze_sales;

table_name,row_count
bronze_customers,40000
bronze_products,2000
bronze_sales,250000
